### Use the published autochem workflow to generate DFT features for the compounds in the dataset

NOTE: This notebook needs to be run with the python environment for autoqchem.

In [1]:
from autoqchem.molecule import molecule
from autoqchem.sge_manager import sge_manager
from autoqchem.draw_utils import draw
from autoqchem.db_functions import descriptors
from rdkit import Chem
import pandas as pd
import numpy as np
import logging
logging.basicConfig(level=logging.INFO)


In [ ]:
# connect to UCLA's computation cluster
sm = sge_manager(user='XXXXXX', host='hoffman2.idre.ucla.edu')
sm.connect()

### Load the product smiles strings and create input files for the DFT calculations

In [7]:
# Load the previously saved list of reaction products
df = pd.read_csv("./cho_processed_data.csv",index_col=0)
df

,conv,deltaG
reactant_aldehyde_nostereo,,
C#CCOc1ccccc1C=O,89.5,1.363565
C#Cc1cc(F)ccc1C=O,19.0,1.044978
C#Cc1ccc(C=O)nc1,98.0,0.901573
C#Cc1ccc(C=O)o1,99.0,1.387973
C#Cc1cccc(C=O)c1,99.0,1.387973
...,...,...
O=Cc1sc2ccccc2c1Br,99.0,0.830152
O=Cc1scc(Br)c1Br,97.5,0.787916
O=Cc1scc2c1OCCO2,21.0,0.990725


In [44]:
aldehyde_smiles = df.index.to_list()
print(f"Total number of substrates: {len(aldehyde_smiles)}")

Total number of substrates: 1091


As the dataset only contains aromatic aldehydes, we decided to limit the number of conformers to a maximum of 5.

In [45]:
# generate molecule objects with up to 5 conformers for each structure
mols = [molecule(s, num_conf=5) for s in aldehyde_smiles]

[16:14:44] WARNING: Charges were rearranged

[16:14:45] WARNING: Charges were rearranged

[16:14:45] WARNING: Charges were rearranged

[16:14:45] WARNING: Charges were rearranged

[16:14:45] WARNING: Charges were rearranged

[16:14:45] WARNING: Charges were rearranged

[16:14:45] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:46] WARNING: Charges were rearranged

[16:14:47] WARNING: Charges were rearranged

[16:14:47] WARNING: Charges were rearranged

[16:14:47] WARNING: Charges were rearranged

[16:14:47] WARNING: Charges were rearranged

[16:14:47] WARNING: Charges were rearranged

[16:14:47]

In [ ]:
# check the compounds by drawing them
draw(mols[133].mol)

interactive(children=(Dropdown(description='confId', options=(0, 1, 2, 3, 4), value=0), Output()), _dom_classe…

<function autoqchem.draw_utils._graph_conf(m, confId=0, energies=[])>

In [48]:
# create Gaussian jobs locally
for mol in mols:
    sm.create_jobs_for_molecule(mol, theory="APFD",heavy_basis_set="def2tzvp",light_basis_set='def2svp',max_light_atomic_number=10)

INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 4 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 4 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 3 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 4 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 4 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 3 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 5 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 5 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 4 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input files for 4 conformations.
INFO:autoqchem.gaussian_input_generator:Generating Gaussian input file

### Manage the DFT jobs on the cluster

In [321]:
# Submit jobs
sm.submit_jobs()

INFO:autoqchem.sge_manager:Submitting 0 jobs.


In [344]:
# Resubmit jobs that did not finish properly
sm.resubmit_incomplete_jobs()

INFO:autoqchem.sge_manager:There are no incomplete jobs to resubmit.


In [352]:
# Retrieve finished jobs from the cluster
sm.retrieve_jobs()

INFO:autoqchem.sge_manager:There are no jobs submitted to cluster. Nothing to retrieve.


In [351]:
# Upload data for finished compounds to the autoqchem database (autoqchem.org)
sm.upload_done_molecules_to_db(tags=["SVR_ArCHO"])

INFO:autoqchem.sge_manager:There are 2 finished molecules ['O=Cc1ccc(Br)cc1N1CCCCC1', 'O=Cc1cccc(OC2CCCC2)c1'].
INFO:autoqchem.sge_manager:Molecule O=Cc1ccc(Br)cc1N1CCCCC1 has 1 / 4 duplicate conformers.
INFO:autoqchem.sge_manager:Removing 1 / 4 jobs and log files that contain duplicate conformers.
INFO:autoqchem.sge_manager:Uploaded descriptors to DB for smiles: O=Cc1ccc(Br)cc1N1CCCCC1, number of conformers: 3, DB molecule id 6976db2712d8c3eed8534abf.
INFO:autoqchem.sge_manager:Molecule O=Cc1cccc(OC2CCCC2)c1 has 2 / 5 duplicate conformers.
INFO:autoqchem.sge_manager:Removing 2 / 5 jobs and log files that contain duplicate conformers.
INFO:autoqchem.sge_manager:Uploaded descriptors to DB for smiles: O=Cc1cccc(OC2CCCC2)c1, number of conformers: 3, DB molecule id 6976db3012d8c3eed8534acb.


### Get the desriptors from the autoqchem database

In [8]:
# Download the descriptors
data = descriptors(tags=["SVR_ArCHO"],presets=["global","substructure"],conf_option="boltzmann",solvent="None",
                   functional="APFD",basis_set="def2svp",substructure="c[CH1]=O")

In [9]:
# Process the data so that it is in one dataframe
label_dict={}
for key in data:
    if key != "global":
        # atom descriptor dataframes are by default called atom1, atom2, etc. --> replace with the atom type and a running number (e. g. "C1" and "C2")
        if data[key].iloc[0,-1] not in label_dict:
            label_dict[data[key].iloc[0,-1]] = 1
        else:
            label_dict[data[key].iloc[0,-1]] += 1
        label = data[key].iloc[0,-1]+str(label_dict[data[key].iloc[0,-1]])
        data[key].drop(columns=["labels","X","Y","Z"],inplace=True)
        data[key].columns = [f"{label}_{column}" for column in data[key].columns]
    else:
        data[key].drop(columns=["converged","multiplicity"],inplace=True)

df_combined = pd.concat(data,axis=1)
df_combined.columns = [multi_column_index[1] for multi_column_index in df_combined.columns]

In [10]:
df_combined

,E,ES_root_dipole,ES_root_electronic_spatial_extent,ES_root_molar_volume,E_scf,E_thermal_correction,E_zpe,G,G_thermal_correction,H,...,O1_ES_root_NPA_valence,O1_Mulliken_charge,O1_NMR_anisotropy,O1_NMR_shift,O1_NPA_Rydberg,O1_NPA_charge,O1_NPA_core,O1_NPA_total,O1_NPA_valence,O1_VBur
can,,,,,,,,,,,,,,,,,,,,,
C#CCOc1ccccc1C=O,-535.241046,2.412503,2150.998995,1489.215379,-535.401038,0.163463,-535.251681,-535.288830,0.115678,-535.240102,...,6.243804,-0.223476,1058.189778,-282.911389,0.011326,-0.543163,1.99975,8.543163,6.532086,0.345831
C#Cc1cc(F)ccc1C=O,-520.033273,1.804206,1771.843809,1192.092478,-520.152134,0.120799,-520.042395,-520.076885,0.077186,-520.032329,...,6.242814,-0.210899,1041.636997,-288.731291,0.011483,-0.533002,1.99975,8.533002,6.52177,0.343764
C#Cc1ccc(C=O)nc1,-436.956579,0.773308,1639.936915,1083.076225,-437.068942,0.115857,-436.964836,-436.998176,0.074261,-436.955635,...,6.273134,-0.200917,1052.11461,-306.856389,0.01142,-0.525725,1.99975,8.525725,6.514554,0.339353
C#Cc1ccc(C=O)o1,-418.749763,0.872910,1305.402319,1192.536914,-418.844997,0.097041,-418.757291,-418.789629,0.057175,-418.748819,...,6.222069,-0.212529,1034.1606,-279.886726,0.011603,-0.525391,1.99975,8.525391,6.514029,0.326561
C#Cc1cccc(C=O)c1,-420.933863,1.625081,1583.477871,1089.313560,-421.060079,0.127998,-420.942200,-420.975584,0.086277,-420.932918,...,6.228519,-0.207329,1026.393022,-291.740615,0.011547,-0.527765,1.99974,8.527765,6.516471,0.340595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
O=Cc1sc2ccccc2c1Br,-3392.524186,0.627590,3027.780329,1706.535763,-3392.644401,0.124580,-3392.534202,-3392.571191,0.077576,-3392.523241,...,6.226455,-0.209598,1015.807763,-287.566331,0.011772,-0.528016,1.99975,8.528016,6.516504,0.344592
O=Cc1scc(Br)c1Br,-5812.547205,2.373500,2657.675900,1266.889000,-5812.610568,0.065653,-5812.556044,-5812.592621,0.020237,-5812.546260,...,6.23929,-0.188702,1022.0061,-277.5685,0.01205,-0.51491,1.99974,8.51491,6.50311,0.367841
O=Cc1scc2c1OCCO2,-893.212341,1.255300,1813.319650,1406.930000,-893.341021,0.132033,-893.221295,-893.255940,0.088434,-893.211397,...,6.22794,-0.209676,995.3144,-266.906,0.01153,-0.53188,1.99974,8.53188,6.5206,0.346995


In [11]:
def feature_preprocessing(df):
    """
    Function for removing non-varied and highly correlated features.
    Take a df as input and returns it in processed form.
    """
    # Remove columns that have only one unique value.
    removed_columns = []
    for column in df.columns:
        if len(np.unique(df[column].values)) < 2:
            removed_columns.append(column)
    df = df.drop(removed_columns, axis=1)
    
    # Remove highly correlated features
    corr_matrix = df.corr().abs()
    upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape),k=1).astype(bool))
    to_drop = [column for column in upper.columns if any(upper[column] > 0.95)]
    df = df.drop(to_drop, axis=1)

    # Store the names of the column removed due to correlation
    for column in to_drop:
        removed_columns.append(column)

    print(f"The following features were removed: {removed_columns}")
    
    return df

df_processed = feature_preprocessing(df_combined)
df_processed

The following features were removed: ['charge', 'E_scf', 'E_zpe', 'G', 'G_thermal_correction', 'H', 'H_thermal_correction', 'electronic_spatial_extent', 'number_of_atoms', 'zero_point_correction', 'C1_ES_root_NPA_total', 'C1_ES_root_NPA_valence', 'C1_Mulliken_charge', 'C1_NPA_Rydberg', 'C1_NPA_charge', 'C1_NPA_core', 'C1_NPA_total', 'C1_NPA_valence', 'C2_ES_root_NPA_total', 'C2_ES_root_NPA_valence', 'C2_NPA_total', 'C2_NPA_valence', 'C2_VBur', 'O1_ES_root_NPA_charge', 'O1_ES_root_NPA_total', 'O1_ES_root_NPA_valence', 'O1_NPA_charge', 'O1_NPA_total', 'O1_NPA_valence']


,E,ES_root_dipole,ES_root_electronic_spatial_extent,ES_root_molar_volume,E_thermal_correction,dipole,electronegativity,hardness,homo_energy,lumo_energy,...,O1_APT_charge,O1_ES_root_Mulliken_charge,O1_ES_root_NPA_Rydberg,O1_ES_root_NPA_core,O1_Mulliken_charge,O1_NMR_anisotropy,O1_NMR_shift,O1_NPA_Rydberg,O1_NPA_core,O1_VBur
can,,,,,,,,,,,,,,,,,,,,,
C#CCOc1ccccc1C=O,-535.241046,2.412503,2150.998995,1489.215379,0.163463,4.528223,0.157789,0.095037,-0.252826,-0.062751,...,-0.71703,-0.068724,0.012028,1.99975,-0.223476,1058.189778,-282.911389,0.011326,1.99975,0.345831
C#Cc1cc(F)ccc1C=O,-520.033273,1.804206,1771.843809,1192.092478,0.120799,2.147358,0.175226,0.094974,-0.270200,-0.080252,...,-0.695181,-0.060929,0.01187,1.99975,-0.210899,1041.636997,-288.731291,0.011483,1.99975,0.343764
C#Cc1ccc(C=O)nc1,-436.956579,0.773308,1639.936915,1083.076225,0.115857,3.449468,0.182819,0.089309,-0.272128,-0.093510,...,-0.711702,-0.070252,0.011691,1.99975,-0.200917,1052.11461,-306.856389,0.01142,1.99975,0.339353
C#Cc1ccc(C=O)o1,-418.749763,0.872910,1305.402319,1192.536914,0.097041,3.638331,0.167003,0.086197,-0.253201,-0.080806,...,-0.773616,-0.054885,0.012413,1.999759,-0.212529,1034.1606,-279.886726,0.011603,1.99975,0.326561
C#Cc1cccc(C=O)c1,-420.933863,1.625081,1583.477871,1089.313560,0.127998,2.913526,0.168968,0.093309,-0.262277,-0.075659,...,-0.701817,-0.054074,0.012282,1.99975,-0.207329,1026.393022,-291.740615,0.011547,1.99974,0.340595
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
O=Cc1sc2ccccc2c1Br,-3392.524186,0.627590,3027.780329,1706.535763,0.124580,4.080050,0.168890,0.078054,-0.246944,-0.090836,...,-0.739522,-0.053965,0.01212,1.99975,-0.209598,1015.807763,-287.566331,0.011772,1.99975,0.344592
O=Cc1scc(Br)c1Br,-5812.547205,2.373500,2657.675900,1266.889000,0.065653,3.701400,0.179205,0.091185,-0.270390,-0.088020,...,-0.65019,-0.05035,0.01223,1.99975,-0.188702,1022.0061,-277.5685,0.01205,1.99974,0.367841
O=Cc1scc2c1OCCO2,-893.212341,1.255300,1813.319650,1406.930000,0.132033,4.158700,0.149740,0.089490,-0.239230,-0.060250,...,-0.706553,-0.052882,0.01228,1.99975,-0.209676,995.3144,-266.906,0.01153,1.99974,0.346995


In [12]:
# make sure that the smiles are canonical in both dataframes (features and labels)
df.index = [Chem.MolToSmiles(Chem.MolFromSmiles(smiles),canonical=True) for smiles in df.index]
df_processed.index = [Chem.MolToSmiles(Chem.MolFromSmiles(smiles),canonical=True) for smiles in df_processed.index]

# map the labels
df_processed["conversion"] = df_processed.index.map(df["conv"])
df_processed["selectivity"] = df_processed.index.map(df["deltaG"])

In [15]:
# check that all yields were assigned
print("Compounds without a conversion:",len(df_processed[df_processed["conversion"].isna()]))
print("Compounds without a selectivity:",len(df_processed[df_processed["selectivity"].isna()]))

Compounds without a conversion: 0
Compounds without a selectivity: 0


In [14]:
# Save the dataset
df_processed.to_csv("./cho_dset.csv",index=True,header=True)